## Architecture Overview:
An input image is processed by a Swin Transformer to produce a set of visual tokens representing different spatial regions. In parallel, tabular features are encoded using an MLP to obtain a compact embedding. The tabular embedding is then used to query the visual tokens through a cross-attention mechanism, allowing the model to selectively focus on relevant image regions conditioned on the tabular data. The fused representation is further refined using a Transformer-style block with residual connections and a feed-forward network. In addition, a global image representation is obtained by pooling the visual tokens. The final representation is formed by combining the fused features with this image-only representation, and is passed through a fully connected head to produce the final prediction.

Image -> Visual Encoder (SWIN) -> Tokens (K, V) 

Tabular -> Encoder -> Query (Q)

Q + (K, V) -> Cross-Attention -> Fuse -> + Image Feature -> Prediction

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
sys.path.insert(0, os.path.abspath("../.."))

In [3]:
# Cell 1: imports & setup
import os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gc

gc.enable()

import torch


from src.config import EXP_CONFIGS
from src.utils import set_seed, save_config




main_folder = "../.."
data_csv = os.path.join(main_folder, "data", "train.csv")
img_folder = os.path.join(main_folder, "data", "train")
out_dir = os.path.join(main_folder, "outputs","extra", "exp13_swin_cross_attn_bce_loss_single_block_tab_transformer")

df = pd.read_csv(data_csv)
cfg = EXP_CONFIGS["exp13_cross_attn"]   # define in config
cfg["name"] = "exp13_swin_cross_attn_bce_loss_single_block_tab_transformer"  #
cfg["loss"] = "bce"  #

TARGET = "Pawpularity"
tab_cols = [c for c in df.columns if c not in ["Id", TARGET]]

os.makedirs(out_dir, exist_ok=True)
save_config(cfg, out_dir)
set_seed(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [4]:
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error
from src.train import run_single_fold

kf = KFold(n_splits=cfg["n_splits"], shuffle=True, random_state=cfg["seed"])
oof_pred = np.zeros(len(df))
oof_true = df[TARGET].values
fold_index = np.full(len(df), -1, dtype=int)
fold_rmse = []

start_all = time.time()
for fold, (tr_idx, val_idx) in enumerate(kf.split(df), start=1):
    print(f"\n=== {cfg['name']}: Fold {fold} ===")
    train_df = df.iloc[tr_idx].reset_index(drop=True)
    val_df   = df.iloc[val_idx].reset_index(drop=True)

    best_rmse, val_preds, val_targets, val_ids = run_single_fold(
    fold=fold,
    train_df=train_df,
    val_df=val_df,
    img_folder=img_folder,
    cfg=cfg,
    out_dir=out_dir,
    device=device,
    mode="cross_attn_swin",    
    tab_cols=tab_cols,
    workers=16,
    pin_memory=True,
    persistent_workers=True,
    freeze_backbone=False,              # False => finetune EfficientNet-B1,True -> Freezen backbone entirely
    # cross-attention specific
    cross_attn_num_heads=8,
    cross_attn_query_mode="tab_queries_image",  #Q->tabular_embed, K&V = visual_tokens
    cross_attn_dropout=0.1,
    num_cross_attn_blocks=1,
    tab_encoder_capacity="tab_transformer", 
)

    oof_pred[val_idx] = val_preds
    fold_index[val_idx] = fold
    fold_rmse.append(best_rmse)
    print(f"Fold {fold} best RMSE: {best_rmse:.4f}")

all_sec = time.time() - start_all
print(f"\nTotal training time: {int(all_sec//60)}m {int(all_sec%60)}s")
# final metrics + OOF with fold column
oof_rmse = root_mean_squared_error(oof_true, oof_pred)
fold_rmse = np.array(fold_rmse)
print(f"\nOOF RMSE: {oof_rmse:.4f}")
print(f"Fold RMSEs: {fold_rmse.tolist()}  Mean={fold_rmse.mean():.4f}  Std={fold_rmse.std():.4f}")

oof_df = pd.DataFrame({
    "Id": df["Id"],
    "fold": fold_index,     
    "ytrue": oof_true,
    "oof_pred": oof_pred,
})
oof_df["abs_err"] = (oof_df["ytrue"] - oof_df["oof_pred"]).abs()
oof_df.to_csv(os.path.join(out_dir, "oof_detail.csv"), index=False)
oof_df.sort_values("abs_err", ascending=False).head(50).to_csv(
    os.path.join(out_dir, "top50_errors.csv"), index=False
)

np.save(os.path.join(out_dir, "oof_pred.npy"), oof_pred)
np.save(os.path.join(out_dir, "fold_rmse.npy"), fold_rmse)
with open(os.path.join(out_dir, "metrics.txt"), "w") as f:
    f.write(f"OOF_RMSE: {oof_rmse:.4f}\n")
    f.write(f"Fold_RMSE: {fold_rmse.tolist()}\nMean: {fold_rmse.mean():.4f}\nStd: {fold_rmse.std():.4f}\n")


/home/ghias/miniconda3/envs/rapids-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



=== exp13_swin_cross_attn_bce_loss_single_block_tab_transformer: Fold 1 ===
Epoch 1/10 | Fold 1 | Train[BCE]: Loss=0.6503 | ValRMSE: 18.5488
Epoch 2/10 | Fold 1 | Train[BCE]: Loss=0.6408 | ValRMSE: 18.5335
Epoch 3/10 | Fold 1 | Train[BCE]: Loss=0.6319 | ValRMSE: 18.0865
Epoch 4/10 | Fold 1 | Train[BCE]: Loss=0.6227 | ValRMSE: 18.0092
Epoch 5/10 | Fold 1 | Train[BCE]: Loss=0.6106 | ValRMSE: 18.4942
Epoch 6/10 | Fold 1 | Train[BCE]: Loss=0.6018 | ValRMSE: 18.4610
Epoch 7/10 | Fold 1 | Train[BCE]: Loss=0.5938 | ValRMSE: 18.6107
Epoch 8/10 | Fold 1 | Train[BCE]: Loss=0.5906 | ValRMSE: 18.5425
Epoch 9/10 | Fold 1 | Train[BCE]: Loss=0.5876 | ValRMSE: 18.7848
Early stopping at epoch 9
Fold 1 best RMSE: 18.0092

=== exp13_swin_cross_attn_bce_loss_single_block_tab_transformer: Fold 2 ===
Epoch 1/10 | Fold 2 | Train[BCE]: Loss=0.6490 | ValRMSE: 17.9206
Epoch 2/10 | Fold 2 | Train[BCE]: Loss=0.6390 | ValRMSE: 17.8584
Epoch 3/10 | Fold 2 | Train[BCE]: Loss=0.6317 | ValRMSE: 18.3528
Epoch 4/10 | F

In [5]:

import pandas as pd
df = pd.read_csv(out_dir+"/oof_detail.csv")
fold_rmse = []

for fold in range(1, 6):
    sub = df[df.fold == fold]
    rmse = ((sub.ytrue - sub.oof_pred)**2).mean() ** 0.5
    fold_rmse.append(rmse)
    print(f"Fold {fold} OOF RMSE: {rmse:.4f} (n={len(sub)})")

Fold 1 OOF RMSE: 18.0092 (n=1983)
Fold 2 OOF RMSE: 17.8584 (n=1983)
Fold 3 OOF RMSE: 17.5834 (n=1982)
Fold 4 OOF RMSE: 17.6092 (n=1982)
Fold 5 OOF RMSE: 18.3850 (n=1982)
